# Apparel Material Deduplication System
## Pipeline Step 2: Material-Name-Only Clustering & Master Code Assignment

**Objective:** Cluster and de-duplicate each category partition **strictly on `Material name`** (the `Colour` column is never read, split or scored; no SKU/variant suffix codes are generated).
1. **Normalization**: lowercase, symbol standardisation (`×`->`x`, `&`->`and`), punctuation removal, alphabetical token sort.
2. **Candidate generation**: <= 600 unique names -> all pairs; otherwise token (>= 3 chars) / 4-char-prefix blocking + length filter.
3. **Scoring**: RapidFuzz Levenshtein (fallback `difflib.SequenceMatcher`), threshold >= 0.85.
4. **Clustering**: Union-Find (DSU). Per-category `cluster_id` from 1 and `new_master_code` (e.g. `FAB-000001`).
5. **Export per category**: `data/03_clusters/clustered_<slug>.csv` and `data/04_curated/curated_<slug>.csv`.

In [ ]:
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Set
import pandas as pd
import yaml

try:
    from IPython.display import display
except ImportError:  # running outside Jupyter
    display = print

# -------------------------------------------------------------------------
# 1. Environment & Path Configuration
# -------------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "notebook"] else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OVERVIEWS_DIR = DATA_DIR / "overviews"
PARTITION_DIR = DATA_DIR / "02_by_category"
CLUSTERS_DIR = DATA_DIR / "03_clusters"
CURATED_DIR = DATA_DIR / "04_curated"
CONFIG_DIR = PROJECT_ROOT / "config"

CATEGORIES_OVERVIEW_PATH = OVERVIEWS_DIR / "categories_overview.csv"

# Ensure intermediate and curated output directories exist
for directory in (CLUSTERS_DIR, CURATED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"[SETUP] Project Root:             {PROJECT_ROOT}")
print(f"[SETUP] Categories Overview File: {CATEGORIES_OVERVIEW_PATH}")
print(f"[SETUP] Category Partitions Dir:  {PARTITION_DIR}")
print(f"[SETUP] Clusters Output Dir:      {CLUSTERS_DIR}")
print(f"[SETUP] Curated Output Dir:       {CURATED_DIR}")
print(f"[SETUP] Config Dir:               {CONFIG_DIR}")

In [ ]:
# -------------------------------------------------------------------------
# 2. Configuration Ingestion & Threshold Rules
# -------------------------------------------------------------------------
prefixes_config_path = CONFIG_DIR / "prefixes.yaml"
if prefixes_config_path.exists():
    with open(prefixes_config_path, "r", encoding="utf-8") as f:
        prefixes_cfg = yaml.safe_load(f) or {}
else:
    prefixes_cfg = {
        "prefixes": {"fabric": "FAB", "trim": "TRM", "thread": "THD", "label": "LBL", "packaging": "PKG", "default": "GEN"},
        "code_format": {"pad_length": 6, "separator": "-"}
    }

PREFIX_MAP: Dict[str, str] = prefixes_cfg.get("prefixes", {})
CODE_FORMAT: Dict[str, object] = prefixes_cfg.get("code_format", {})
PAD_LENGTH: int = CODE_FORMAT.get("pad_length", 6)
SEPARATOR: str = CODE_FORMAT.get("separator", "-")
DEFAULT_PREFIX: str = PREFIX_MAP.get("default", "GEN")

# Dynamically load match threshold from config/splink_rules.yaml
splink_config_path = CONFIG_DIR / "splink_rules.yaml"
if splink_config_path.exists():
    with open(splink_config_path, "r", encoding="utf-8") as f:
        splink_cfg = yaml.safe_load(f) or {}
else:
    splink_cfg = {"match_threshold": 0.85}

SIMILARITY_THRESHOLD: float = float(splink_cfg.get("match_threshold", 0.85))

print(f"[CONFIG] Prefix mapping:             {PREFIX_MAP}")
print(f"[CONFIG] Code format:                Pad={PAD_LENGTH}, Separator='{SEPARATOR}', Default='{DEFAULT_PREFIX}'")
print(f"[CONFIG] Loaded from splink_rules:   SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}")

In [ ]:
# -------------------------------------------------------------------------
# 3. Load Valid Categories from categories_overview.csv
# -------------------------------------------------------------------------
if not CATEGORIES_OVERVIEW_PATH.exists():
    raise FileNotFoundError(f"Categories overview file not found: {CATEGORIES_OVERVIEW_PATH}. Please run Step 1 notebook first.")

df_categories_overview = pd.read_csv(CATEGORIES_OVERVIEW_PATH)
print(f"[LOAD] Loaded categories overview from: {CATEGORIES_OVERVIEW_PATH.name}")
display(df_categories_overview)

# Extract valid categories list
valid_categories: List[str] = df_categories_overview["Category"].dropna().unique().tolist()
# Uncategorized rows are exported by Step 1 into their own partition; include them so no record is lost
if (PARTITION_DIR / "category_uncategorized.csv").exists() and "Uncategorized" not in valid_categories:
    valid_categories.append("Uncategorized")
print(f"\n[LOAD] Identified {len(valid_categories)} valid categories to process: {valid_categories}")

In [ ]:
# -------------------------------------------------------------------------
# 4. Material Name Normalization Helper
# -------------------------------------------------------------------------
def normalize_material_name(text: Optional[str]) -> str:
    """
    Normalizes Material name by:
    1. Lowercase
    2. Standard symbol replacement (& -> and, % -> percent, + -> plus, etc.)
    3. Stripping all remaining punctuation
    4. Alphabetical token sorting (eliminating word-order variance)
    """
    if text is None or pd.isna(text):
        return ""
    
    # 1. Lowercase
    cleaned = str(text).lower()
    
    # 2. Standard symbol replacement
    symbol_replacements = {
        "&": " and ",
        "%": " percent ",
        "+": " plus ",
        "@": " at ",
        "#": " number ",
        "/": " ",
        "-": " ",
        "_": " ",
        "×": " x ",
    }
    for symbol, replacement in symbol_replacements.items():
        cleaned = cleaned.replace(symbol, replacement)
        
    # Dimension separator only between digits (e.g. 2x2 -> 2 2); never touch 'x' inside words
    cleaned = re.sub(r"(?<=\d)x(?=\d)", " ", cleaned)
    
    # 3. Remove remaining punctuation and non-alphanumeric chars
    cleaned = re.sub(r"[^\w\s]", " ", cleaned)
    
    # 4. Alphabetical token sorting
    tokens = [tok for tok in cleaned.split() if tok]
    tokens.sort()
    
    return " ".join(tokens)


def normalize_category_slug(category_name: str) -> str:
    """
    Normalizes category name into partition file slug.
    Example: 'Fabric' -> 'fabric', 'Packaging / Boxing' -> 'packaging_boxing'
    """
    if not category_name or pd.isna(category_name):
        return "uncategorized"
    clean = re.sub(r"[^a-zA-Z0-9]+", "_", str(category_name).strip().lower())
    return clean.strip("_") or "uncategorized"


# Normalization verification tests
test_cases = [
    "Cotton single jersey",
    "Single Jersey, Cotton!",
    "Cotton, single-jersey & 100%",
    "Polyester sewing thread (Tex 27)",
    "Nylon coil zipper (No. 5)"
]
print("[TEST] Material name normalization:")
for tc in test_cases:
    print(f"  '{tc}' -> '{normalize_material_name(tc)}'")

In [ ]:
# -------------------------------------------------------------------------
# 5. String Similarity & Union-Find / Connected Components Engine
# -------------------------------------------------------------------------
import difflib

try:
    from rapidfuzz.distance import Levenshtein as rf_levenshtein
    from rapidfuzz import fuzz as rf_fuzz
    def compute_string_similarity(s1: str, s2: str) -> float:
        # names are already token-sorted; take the better of Levenshtein and token_sort_ratio
        if s1 == s2:
            return 1.0
        return max(float(rf_levenshtein.normalized_similarity(s1, s2)), rf_fuzz.token_sort_ratio(s1, s2) / 100.0)
except ImportError:
    def compute_string_similarity(s1: str, s2: str) -> float:
        if s1 == s2:
            return 1.0
        return float(difflib.SequenceMatcher(None, s1, s2).ratio())


class UnionFind:
    """
    Disjoint Set Union (DSU) / Union-Find data structure with
    path compression and rank optimization for clustering connected components.
    """
    def __init__(self, elements):
        self.parent = {x: x for x in elements}
        self.rank = {x: 0 for x in elements}

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x == root_y:
            return
        if self.rank[root_x] < self.rank[root_y]:
            self.parent[root_x] = root_y
        elif self.rank[root_x] > self.rank[root_y]:
            self.parent[root_y] = root_x
        else:
            self.parent[root_y] = root_x
            self.rank[root_x] += 1

    def get_components(self) -> Dict[int, List[int]]:
        """Returns mapping of root_id -> list of element IDs in that cluster."""
        components: Dict[int, List[int]] = {}
        for x in self.parent:
            root = self.find(x)
            components.setdefault(root, []).append(x)
        return components

# Quick sanity test of similarity on normalized material names
s_a = normalize_material_name("Cotton single jersey")
s_b = normalize_material_name("Single Jersey Cotton")
sim_test = compute_string_similarity(s_a, s_b)
print(f"[TEST] String similarity between '{s_a}' and '{s_b}': {sim_test:.4f}")

In [ ]:
# -------------------------------------------------------------------------
# 6. Material-name-only Blocking & Clustering Engine
# -------------------------------------------------------------------------
OUTPUT_COLUMNS = ["Material/item code", "Material name", "norm_material_name", "cluster_id",
                  "new_master_code", "similarity_score", "cluster_record_count", "is_duplicate"]

def deduplicate_and_cluster_partition(df_partition: pd.DataFrame, category_name: str,
                                      threshold: float = SIMILARITY_THRESHOLD) -> pd.DataFrame:
    """Cluster rows of ONE category by normalized Material name only (Colour ignored)."""
    df = df_partition.copy().reset_index(drop=True)
    n = len(df)
    name_col = next((c for c in df.columns if c.strip().lower() in ["material name", "material_name", "item name"]), "Material name")
    code_col = next((c for c in df.columns if c.strip().lower() in ["material/item code", "material item code", "item code"]), None)
    df["Material name"] = df[name_col]
    df["Material/item code"] = df[code_col] if code_col else ""

    norm_lookup = {raw: normalize_material_name(raw) for raw in df["Material name"].unique()}
    df["norm_material_name"] = df["Material name"].map(norm_lookup)

    name_to_rows: Dict[str, List[int]] = {}
    for idx, nm in enumerate(df["norm_material_name"]):
        name_to_rows.setdefault(nm, []).append(idx)
    unique_names = list(name_to_rows.keys())
    m = len(unique_names)

    MAX_BLOCK_SIZE = 300
    candidate_pairs: Set[Tuple[int, int]] = set()
    if m <= 600:
        for i in range(m):
            for j in range(i + 1, m):
                candidate_pairs.add((i, j))
    else:
        block_index: Dict[str, List[int]] = {}
        for u, name in enumerate(unique_names):
            for tok in set(name.split()):
                if len(tok) >= 3:
                    block_index.setdefault("t_" + tok, []).append(u)
            if len(name) >= 4:
                block_index.setdefault("p_" + name[:4], []).append(u)
        for members in block_index.values():
            if 1 < len(members) <= MAX_BLOCK_SIZE:
                for i in range(len(members)):
                    for j in range(i + 1, len(members)):
                        a, b = members[i], members[j]
                        candidate_pairs.add((a, b) if a < b else (b, a))

    uf = UnionFind(range(m))
    for u1, u2 in candidate_pairs:
        s1, s2 = unique_names[u1], unique_names[u2]
        if min(len(s1), len(s2)) < threshold * max(len(s1), len(s2)):
            continue  # length filter
        if compute_string_similarity(s1, s2) >= threshold:
            uf.union(u1, u2)

    components = uf.get_components()
    cat_slug = normalize_category_slug(category_name)
    prefix = PREFIX_MAP.get(cat_slug, PREFIX_MAP.get(str(category_name).lower(), DEFAULT_PREFIX))

    cluster_ids = [0] * n
    scores = [1.0] * n
    master_codes = [""] * n
    counts = [1] * n

    sorted_roots = sorted(components.keys(), key=lambda r: min(components[r]))
    for cid, root in enumerate(sorted_roots, start=1):
        members = components[root]
        rows = [r for u in members for r in name_to_rows[unique_names[u]]]
        code = f"{prefix}{SEPARATOR}{str(cid).zfill(PAD_LENGTH)}"
        # canonical representative: most frequent name (ties -> shortest, then alphabetical)
        rep = min(members, key=lambda u: (-len(name_to_rows[unique_names[u]]), len(unique_names[u]), unique_names[u]))
        for u in members:
            sc = 1.0 if u == rep else round(compute_string_similarity(unique_names[u], unique_names[rep]), 4)
            for r in name_to_rows[unique_names[u]]:
                cluster_ids[r] = cid
                scores[r] = sc
                master_codes[r] = code
                counts[r] = len(rows)

    df["cluster_id"] = cluster_ids
    df["new_master_code"] = master_codes
    df["similarity_score"] = scores
    df["cluster_record_count"] = counts
    df["is_duplicate"] = df["cluster_record_count"] > 1
    return df[OUTPUT_COLUMNS]

In [ ]:
# -------------------------------------------------------------------------
# 7. Per-category processing & export
# -------------------------------------------------------------------------
processed_partitions: List[pd.DataFrame] = []
partition_summary: List[dict] = []
start_time = time.time()

for cat in valid_categories:
    t_cat = time.time()
    cat_slug = normalize_category_slug(cat)
    partition_file = PARTITION_DIR / f"category_{cat_slug}.csv"
    if not partition_file.exists():
        print(f"  [WARN] Partition file not found: {partition_file.name} for category '{cat}'")
        continue

    df_part = pd.read_csv(partition_file)
    raw_count = len(df_part)
    df_clustered = deduplicate_and_cluster_partition(df_part, cat, SIMILARITY_THRESHOLD)

    cluster_path = CLUSTERS_DIR / f"clustered_{cat_slug}.csv"
    curated_path = CURATED_DIR / f"curated_{cat_slug}.csv"
    df_clustered.to_csv(cluster_path, index=False)
    df_clustered.sort_values(["cluster_id", "Material/item code"]).to_csv(curated_path, index=False)
    processed_partitions.append(df_clustered.assign(Category=cat))

    unique_clusters = df_clustered["cluster_id"].nunique()
    dupes_count = int(df_clustered["is_duplicate"].sum())
    elapsed = time.time() - t_cat
    partition_summary.append({"Category": cat, "Raw Records": raw_count, "Unique Clusters": unique_clusters,
                              "Duplicate Rows": dupes_count, "Elapsed (s)": f"{elapsed:.2f}s",
                              "Clustered File": cluster_path.name, "Curated File": curated_path.name})
    print(f"  [OK] {cat:<16} | Raw: {raw_count:>6,} | Clusters: {unique_clusters:>5} | Duplicates: {dupes_count:>6,} | {elapsed:>5.2f}s")

print(f"\n[PERFORMANCE] Completed in {time.time() - start_time:.2f} seconds")
df_partition_summary = pd.DataFrame(partition_summary)
display(df_partition_summary)

In [ ]:
# -------------------------------------------------------------------------
# 8. Combined view (in-memory) for metrics only; per-category files already written
# -------------------------------------------------------------------------
if not processed_partitions:
    raise RuntimeError("No category partitions were processed. Please check input files.")
df_master_dedup = pd.concat(processed_partitions, ignore_index=True)
df_duplicates_audit = df_master_dedup[df_master_dedup["is_duplicate"]].copy()
print(f"[EXPORT] {len(processed_partitions)} clustered files -> {CLUSTERS_DIR}")
print(f"[EXPORT] {len(processed_partitions)} curated files   -> {CURATED_DIR}")

In [ ]:
# -------------------------------------------------------------------------
# 9. Pipeline Metrics
# -------------------------------------------------------------------------
total_records = len(df_master_dedup)
total_clusters = len(df_master_dedup.groupby(["Category", "cluster_id"]))
duplicate_records = len(df_duplicates_audit)
print("=" * 75)
print(f"Total Records Processed:   {total_records:,}")
print(f"Total Clusters (per cat.): {total_clusters:,}")
print(f"Duplicate Rows Flagged:    {duplicate_records:,} ({duplicate_records / total_records * 100:.1f}%)")
print(f"Similarity Threshold:      {SIMILARITY_THRESHOLD}")
print("=" * 75)
if not df_duplicates_audit.empty:
    display(df_duplicates_audit.sort_values(["Category", "cluster_id"]).head(10))

## 10. Visual Summary  <!-- VIS_CELL -->
Compression achieved per category, cluster sizes and similarity scores.

In [ ]:
# VIS_CELL
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})
PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974", "#64B5CD", "#937860"]

summ = df_partition_summary.copy()
summ["Compression %"] = ((1 - summ["Unique Clusters"] / summ["Raw Records"]) * 100).round(2)
display(summ[["Category", "Raw Records", "Unique Clusters", "Duplicate Rows", "Compression %", "Elapsed (s)"]])

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
x = range(len(summ)); w = 0.4
ax[0].bar([i - w/2 for i in x], summ["Raw Records"], w, label="Raw records", color=PALETTE[0])
ax[0].bar([i + w/2 for i in x], summ["Unique Clusters"], w, label="Unique clusters", color=PALETTE[1])
ax[0].set_yscale("log"); ax[0].set_xticks(list(x)); ax[0].set_xticklabels(summ["Category"], rotation=30)
ax[0].set_title("Raw records vs unique clusters (log scale)"); ax[0].legend()

sizes = df_master_dedup.groupby(["Category", "cluster_id"]).size().sort_values(ascending=False)
ax[1].hist(sizes, bins=30, color=PALETTE[3])
ax[1].set_title("Cluster size distribution (rows per cluster)")
ax[1].set_xlabel("Rows in cluster"); ax[1].set_ylabel("Number of clusters")

ax[2].hist(df_master_dedup["similarity_score"], bins=20, color=PALETTE[2])
ax[2].set_yscale("log"); ax[2].set_title("Similarity score to cluster representative")
ax[2].set_xlabel("Score"); ax[2].set_ylabel("Rows (log)")
plt.tight_layout(); plt.show()

print("Top 10 largest clusters:")
display(sizes.head(10).rename("rows").reset_index())
